# Disease Classification — Full Training Pipeline
**Dataset**: Respiratory disease audio + clinical features (asthma, copd, covid, healthy)  
**Models**: Logistic Regression · Random Forest · Gradient Boosting · HistGradientBoosting · SVM  
**Outputs**: Performance matrix · Confusion matrices · Loss curves

## 1 — Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, joblib
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection  import StratifiedKFold, cross_validate
from sklearn.preprocessing    import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute           import SimpleImputer
from sklearn.pipeline         import Pipeline
from sklearn.compose          import ColumnTransformer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif

from sklearn.linear_model  import LogisticRegression
from sklearn.ensemble      import (RandomForestClassifier,
                                    GradientBoostingClassifier,
                                    HistGradientBoostingClassifier)
from sklearn.svm           import SVC

from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, log_loss,
                              accuracy_score, f1_score)

print("All imports successful ✓")

## 2 — Load Data

In [ ]:
# ── Paths (update if running locally) ──────────────────────────────────
TRAIN_PATH = "train_66_ros.xlsx"
TEST_PATH  = "test_66.csv"
VAL_PATH   = "val_66__1_.csv"
# ───────────────────────────────────────────────────────────────────────

df_train = pd.read_excel(TRAIN_PATH, header=1)   # row-0 is actual header
df_test  = pd.read_csv(TEST_PATH)
df_val   = pd.read_csv(VAL_PATH)

print(f"Train : {df_train.shape}")
print(f"Val   : {df_val.shape}")
print(f"Test  : {df_test.shape}")

print("\nClass distribution (train):")
print(df_train["disease_y"].value_counts().to_string())

## 3 — Feature / Target Split

In [ ]:
TARGET    = "disease_y"
FEAT_COLS = [c for c in df_test.columns if c not in ("id", "disease")]

X_train = df_train[FEAT_COLS].copy()
y_train_raw = df_train[TARGET].copy()

X_test = df_test[FEAT_COLS].copy()
y_test_raw = df_test["disease"].copy()

X_val = df_val[FEAT_COLS].copy()
y_val_raw = df_val["disease"].copy()

print(f"Feature columns : {len(FEAT_COLS)}")
print(f"X_train : {X_train.shape}  |  X_val : {X_val.shape}  |  X_test : {X_test.shape}")

## 4 — Column-type Detection & Boolean Fix

In [ ]:
bool_cols = X_train.select_dtypes(include="bool").columns.tolist()
cat_cols  = X_train.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols  = X_train.select_dtypes(include=np.number).columns.tolist()

# Convert bool → float for all splits
for df in [X_train, X_val, X_test]:
    df[bool_cols] = df[bool_cols].astype(float)

for c in bool_cols:
    if c not in num_cols:
        num_cols.append(c)

print(f"Numeric cols  : {len(num_cols)}")
print(f"Categorical   : {cat_cols}")
print(f"Boolean fixed : {bool_cols}")

## 5 — Label Encoding

In [ ]:
le = LabelEncoder()
le.fit(pd.concat([y_train_raw, y_test_raw, y_val_raw]))

y_train = le.transform(y_train_raw)
y_test  = le.transform(y_test_raw)
y_val   = le.transform(y_val_raw)

print("Class → Integer mapping:")
for cls, idx in zip(le.classes_, range(len(le.classes_))):
    print(f"  {cls:10s} → {idx}")

## 6 — Preprocessing Pipeline

In [ ]:
num_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

cat_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer([
    ("num", num_transformer, num_cols),
    ("cat", cat_transformer, cat_cols),
], remainder="drop")

feature_sel = Pipeline([
    ("var",   VarianceThreshold(0.01)),
    ("kbest", SelectKBest(f_classif, k=40)),
])

def build_pipe(clf):
    return Pipeline([
        ("prep",    preprocessor),
        ("featsel", feature_sel),
        ("clf",     clf),
    ])

print("Pipeline builder ready ✓")

## 7 — Define Models

In [ ]:
MODELS = {
    "LogisticRegression": build_pipe(
        LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs",
                           multi_class="auto", random_state=42)
    ),
    "RandomForest": build_pipe(
        RandomForestClassifier(n_estimators=200, max_depth=None,
                               min_samples_leaf=2, random_state=42, n_jobs=-1)
    ),
    "GradientBoosting": build_pipe(
        GradientBoostingClassifier(n_estimators=150, learning_rate=0.1,
                                   max_depth=4, random_state=42)
    ),
    "HistGradientBoosting": build_pipe(
        HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1,
                                       max_depth=None, random_state=42)
    ),
    "SVM": build_pipe(
        SVC(C=1.0, kernel="rbf", gamma="scale",
            probability=True, random_state=42)
    ),
}

print(f"Models defined: {list(MODELS.keys())}")

## 8 — Train All Models & Collect Metrics

In [ ]:
results = {}

for name, pipe in MODELS.items():
    print(f"\n{'='*55}")
    print(f"  Training: {name}")
    print(f"{'='*55}")

    # Fit on train
    pipe.fit(X_train, y_train)

    # Predictions
    y_val_pred   = pipe.predict(X_val)
    y_test_pred  = pipe.predict(X_test)
    y_val_proba  = pipe.predict_proba(X_val)
    y_test_proba = pipe.predict_proba(X_test)

    # Metrics
    val_acc  = accuracy_score(y_val,  y_val_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    val_f1   = f1_score(y_val,  y_val_pred,  average="weighted")
    test_f1  = f1_score(y_test, y_test_pred, average="weighted")
    val_loss = log_loss(y_val,  y_val_proba)
    test_loss= log_loss(y_test, y_test_proba)

    results[name] = {
        "pipe"         : pipe,
        "y_val_pred"   : y_val_pred,
        "y_test_pred"  : y_test_pred,
        "y_val_proba"  : y_val_proba,
        "y_test_proba" : y_test_proba,
        "val_acc"      : val_acc,
        "test_acc"     : test_acc,
        "val_f1"       : val_f1,
        "test_f1"      : test_f1,
        "val_loss"     : val_loss,
        "test_loss"    : test_loss,
    }

    print(f"  Val  Acc={val_acc:.4f}  F1={val_f1:.4f}  LogLoss={val_loss:.4f}")
    print(f"  Test Acc={test_acc:.4f}  F1={test_f1:.4f}  LogLoss={test_loss:.4f}")

print("\n✓ All models trained.")

## 9 — Performance Matrix (Summary Table)

In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        "Model"        : name,
        "Val Acc"      : round(r["val_acc"],  4),
        "Test Acc"     : round(r["test_acc"], 4),
        "Val F1 (W)"   : round(r["val_f1"],  4),
        "Test F1 (W)"  : round(r["test_f1"], 4),
        "Val LogLoss"  : round(r["val_loss"], 4),
        "Test LogLoss" : round(r["test_loss"],4),
    })

perf_df = pd.DataFrame(rows).set_index("Model")

# Pretty display
styled = perf_df.style\
    .background_gradient(cmap="YlGn",  subset=["Val Acc","Test Acc","Val F1 (W)","Test F1 (W)"])\
    .background_gradient(cmap="YlOrRd_r", subset=["Val LogLoss","Test LogLoss"])\
    .format("{:.4f}")

print("Performance Matrix:")
print(perf_df.to_string())
styled

## 10 — Detailed Classification Report (Best Model on Test)

In [ ]:
best_name = max(results, key=lambda k: results[k]["test_acc"])
print(f"Best model by Test Accuracy: {best_name}")
print("\nClassification Report — TEST set:")
print(classification_report(
    y_test, results[best_name]["y_test_pred"],
    target_names=le.classes_
))

## 11 — Confusion Matrices (Val + Test, All Models)

In [ ]:
n_models = len(MODELS)
fig, axes = plt.subplots(n_models, 2, figsize=(14, 4.5 * n_models))
fig.suptitle("Confusion Matrices — Val (left)  |  Test (right)",
             fontsize=15, fontweight="bold", y=1.01)

for row_idx, (name, r) in enumerate(results.items()):
    for col_idx, (y_true, y_pred, split) in enumerate([
        (y_val,  r["y_val_pred"],  "Val"),
        (y_test, r["y_test_pred"], "Test"),
    ]):
        ax = axes[row_idx][col_idx]
        cm = confusion_matrix(y_true, y_pred)
        disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
        disp.plot(ax=ax, colorbar=False, cmap="Blues", xticks_rotation=30)
        acc = accuracy_score(y_true, y_pred)
        ax.set_title(f"{name} — {split}  (Acc {acc:.3f})", fontsize=11)

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved → confusion_matrices.png")

## 12 — Log-Loss Bar Chart (Val vs Test per Model)

In [ ]:
model_names = list(results.keys())
val_losses  = [results[m]["val_loss"]  for m in model_names]
test_losses = [results[m]["test_loss"] for m in model_names]

x = np.arange(len(model_names))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - w/2, val_losses,  w, label="Val",  color="#4C9BE8", edgecolor="white")
bars2 = ax.bar(x + w/2, test_losses, w, label="Test", color="#E87C4C", edgecolor="white")

ax.bar_label(bars1, fmt="%.3f", padding=3, fontsize=9)
ax.bar_label(bars2, fmt="%.3f", padding=3, fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15, ha="right")
ax.set_ylabel("Log Loss  (lower = better)", fontsize=11)
ax.set_title("Log Loss per Model — Validation vs Test", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
sns.despine()
plt.tight_layout()
plt.savefig("loss_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved → loss_comparison.png")

## 13 — Accuracy & F1 Comparison Plot

In [ ]:
val_accs  = [results[m]["val_acc"]  for m in model_names]
test_accs = [results[m]["test_acc"] for m in model_names]
val_f1s   = [results[m]["val_f1"]   for m in model_names]
test_f1s  = [results[m]["test_f1"]  for m in model_names]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Accuracy ──
ax = axes[0]
x  = np.arange(len(model_names))
b1 = ax.bar(x - w/2, val_accs,  w, label="Val",  color="#4C9BE8", edgecolor="white")
b2 = ax.bar(x + w/2, test_accs, w, label="Test", color="#E87C4C", edgecolor="white")
ax.bar_label(b1, fmt="%.3f", padding=3, fontsize=8)
ax.bar_label(b2, fmt="%.3f", padding=3, fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(model_names, rotation=15, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Accuracy"); ax.set_title("Accuracy per Model", fontweight="bold")
ax.legend(); ax.grid(axis="y", alpha=0.3); sns.despine(ax=ax)

# ── F1 ──
ax = axes[1]
b3 = ax.bar(x - w/2, val_f1s,  w, label="Val",  color="#52C57A", edgecolor="white")
b4 = ax.bar(x + w/2, test_f1s, w, label="Test", color="#C5527A", edgecolor="white")
ax.bar_label(b3, fmt="%.3f", padding=3, fontsize=8)
ax.bar_label(b4, fmt="%.3f", padding=3, fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(model_names, rotation=15, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Weighted F1"); ax.set_title("Weighted F1 per Model", fontweight="bold")
ax.legend(); ax.grid(axis="y", alpha=0.3); sns.despine(ax=ax)

plt.suptitle("Model Comparison — Accuracy & F1", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("accuracy_f1_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved → accuracy_f1_comparison.png")

## 14 — Simulated Training Loss Curve (GradientBoosting)

In [ ]:
# GradientBoostingClassifier tracks oob_improvement_ / train_score_
# We retrain a standalone GBC (no pipeline wrapping) on the preprocessed data
# so we can access per-iteration train_score_ and staged_predict_proba for val.

# --- refit preprocessor+featsel only to get transformed arrays ---
prep_pipe = Pipeline([
    ("prep",    preprocessor),
    ("featsel", feature_sel),
])
X_tr_t = prep_pipe.fit_transform(X_train, y_train)
X_vl_t = prep_pipe.transform(X_val)

gbc = GradientBoostingClassifier(n_estimators=150, learning_rate=0.1,
                                  max_depth=4, random_state=42)
gbc.fit(X_tr_t, y_train)

# Train loss (deviance) per iteration — stored internally
train_scores = gbc.train_score_   # shape (n_estimators,)

# Val loss via staged_predict_proba
val_log_losses = []
for staged_proba in gbc.staged_predict_proba(X_vl_t):
    val_log_losses.append(log_loss(y_val, staged_proba))

iters = np.arange(1, len(train_scores) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Train deviance ──
ax = axes[0]
ax.plot(iters, train_scores, color="#4C9BE8", linewidth=2, label="Train Deviance")
ax.set_xlabel("Boosting Iterations"); ax.set_ylabel("Deviance (lower = better)")
ax.set_title("GradientBoosting — Train Deviance", fontweight="bold")
ax.legend(); ax.grid(alpha=0.3); sns.despine(ax=ax)

# ── Val log-loss ──
ax = axes[1]
ax.plot(iters, val_log_losses, color="#E87C4C", linewidth=2, label="Val Log-Loss")
best_iter = np.argmin(val_log_losses) + 1
ax.axvline(best_iter, color="gray", linestyle="--", alpha=0.7,
           label=f"Best iter = {best_iter}")
ax.set_xlabel("Boosting Iterations"); ax.set_ylabel("Log Loss")
ax.set_title("GradientBoosting — Val Log-Loss Curve", fontweight="bold")
ax.legend(); ax.grid(alpha=0.3); sns.despine(ax=ax)

plt.suptitle("GradientBoosting Loss Curves", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("loss_curves_gb.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved → loss_curves_gb.png  |  Best iteration = {best_iter}")

## 15 — Save Best Model

In [ ]:
best_name = max(results, key=lambda k: results[k]["test_acc"])
best_pipe = results[best_name]["pipe"]

joblib.dump(best_pipe, "best_model.joblib")
joblib.dump(le,        "label_encoder.joblib")

print(f"Best model : {best_name}")
print(f"Test Acc   : {results[best_name]['test_acc']:.4f}")
print(f"Test F1    : {results[best_name]['test_f1']:.4f}")
print("Saved → best_model.joblib, label_encoder.joblib")

## 16 — Inference Example

In [ ]:
# Load and predict on a new sample
loaded_model = joblib.load("best_model.joblib")
loaded_le    = joblib.load("label_encoder.joblib")

sample = X_test.iloc[:5]
preds  = loaded_le.inverse_transform(loaded_model.predict(sample))
proba  = loaded_model.predict_proba(sample)

result_df = pd.DataFrame(proba, columns=[f"P({c})" for c in loaded_le.classes_])
result_df.insert(0, "Predicted", preds)
result_df.insert(0, "True", le.inverse_transform(y_test[:5]))
print(result_df.to_string(index=False))